## 🎯 ML Business Problem Statement
> The goal of this project is to predict salary trends in government jobs using features such as education level, sector, and role category.

- The model helps estimate expected salary ranges for government jobs based on role and qualification patterns. This can support career planning, job market analysis, and data-driven decision making for job seekers and career platforms.
----
### End Clients for Project

**1️⃣ Job Seekers / Students**
- estimate salary expectations
- compare sectors
- understand role trends

**2️⃣ Career Guidance Platforms**
- suggest realistic salary ranges
- guide users toward sectors

**3️⃣ Recruitment Analytics Teams**
- analyze salary structures
- study market patterns

**4️⃣ Educational/Career Institutions**
- understand demand trends
- align training with job market

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [45]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import BernoulliNB,CategoricalNB
from sklearn.linear_model import LogisticRegression,SGDClassifier
from sklearn.multiclass import OneVsRestClassifier # wrapper over Logistic Regression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.ensemble import GradientBoostingClassifier
from xgboost import XGBClassifier
from sklearn.svm import SVC

In [3]:
from sklearn.metrics import confusion_matrix,classification_report
from sklearn.metrics import log_loss
from sklearn.metrics import roc_curve,precision_recall_curve

from sklearn.preprocessing import LabelEncoder,OneHotEncoder,OrdinalEncoder


from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

In [4]:
from sklearn.model_selection import train_test_split

### Data Collection

In [5]:
df = pd.read_csv("../Data/Job_Data_Trimmed.csv")

In [6]:
df.head()

,Job Type,salary_minimum,salary_maximum,salary_band,salary_range,qualification_level,sector,role_category
0,Government,63000.0,63000.0,Medium,0.0,undergraduate,Banking,Officer
1,Government,25000.0,25000.0,Low,0.0,undergraduate,Government General,Associate
2,Government,30000.0,60000.0,Medium,30000.0,undergraduate,PSU,Other
3,Government,57700.0,182400.0,Medium,124700.0,postgraduate,Government Admin,Academic
4,Government,25000.0,25000.0,Low,0.0,undergraduate,PSU,Engineer


### Feature Selection And Preprocessing
----
As of the Problem Statement is regarding the Regression 

- We choose Specific Feature for the Pattern recognition
- Selected Features are
    - Role Category
    - Qualification Level
    - Sector 

In [7]:
feature = df[["role_category","qualification_level","sector","salary_band"]]

In [8]:
feature.isna().sum()

role_category          0
qualification_level    0
sector                 0
salary_band            0
dtype: int64

In [21]:
feature["salary_band"].unique()

array(['Medium', 'Low', 'High', 'Very High'], dtype=object)

#### Spliting and Encoding Flow 
----
1. Separate X and y
2. Train-test split
3. Fit encoders/scalers ONLY on training data
4. Transform training data
5. Transform test data using fitted transformers

In [22]:
# 1. Seperate the X and y
X = df[["qualification_level", "sector", "role_category"]] 
y = df["salary_band"].map({"Low":0,"Medium":1,"High":2,"Very High":3})

# 2. Splitting data
X_train, X_val, y_train, y_val = train_test_split(X, y, train_size = 0.8,test_size=0.2, random_state=42)

In [23]:
X_train["qualification_level"].unique()

array(['undergraduate', 'postgraduate', 'diploma', 'doctorate', 'other',
       'school_level'], dtype=object)

In [24]:
print("X_train : ",len(X_train))
print("y_val : ",len(y_train))
print("X_test : ",len(X_val))
print("y_val : ",len(y_val))

X_train :  521
y_val :  521
X_test :  131
y_val :  131


### Model Training

In [25]:

ordinal_cols = ["qualification_level"] # adding the salary band to the data for performance increasal
nominal_cols = ["sector","role_category"]

In [26]:
# 3. Encoding using the transformer

preprocessor = ColumnTransformer(transformers=[("qual_encoder", OrdinalEncoder(), ordinal_cols),
                                               ("onehot", OneHotEncoder(handle_unknown="ignore"), nominal_cols)])

#### KNN Classifier 

In [27]:
# 4. Intializing the KNN Classifier Model model into the pipeline

# 1. KNN Classifier model  
pipeline = Pipeline(steps=[("preprocessing", preprocessor),
                           ("model", KNeighborsClassifier())])



In [29]:
pipeline.fit(X_train, y_train)

,steps,"[('preprocessing', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('qual_encoder', ...), ('onehot', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


### Predictions

In [30]:
train_pred = pipeline.predict(X_train)
test_pred = pipeline.predict(X_val)

### Validation 

In [31]:
# Validation Data

print("Validation Data loss")
print("--"*20)
print(confusion_matrix(y_val,test_pred))
print(classification_report(y_train,train_pred))
print()


# training data

print("Training Data loss")
print("--"*20)
print(confusion_matrix(y_train,train_pred))
print(classification_report(y_train,train_pred))



Validation Data loss
----------------------------------------
[[23 17  1  0]
 [25 38  4  0]
 [ 5 10  3  0]
 [ 2  1  2  0]]
              precision    recall  f1-score   support

           0       0.52      0.69      0.59       163
           1       0.58      0.67      0.62       242
           2       0.63      0.19      0.29       101
           3       0.00      0.00      0.00        15

    accuracy                           0.56       521
   macro avg       0.43      0.39      0.38       521
weighted avg       0.56      0.56      0.53       521


Training Data loss
----------------------------------------
[[112  46   5   0]
 [ 76 161   5   0]
 [ 22  60  19   0]
 [  5   9   1   0]]
              precision    recall  f1-score   support

           0       0.52      0.69      0.59       163
           1       0.58      0.67      0.62       242
           2       0.63      0.19      0.29       101
           3       0.00      0.00      0.00        15

    accuracy                    

D:\Dev\Envs\batch468\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
D:\Dev\Envs\batch468\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
D:\Dev\Envs\batch468\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
D:\Dev\Envs\batch468\Lib\site-packages\s

In [60]:
# Basic preprocessing 
# 1. Seperate the X and y
X = df[["qualification_level", "sector", "role_category"]] 
y = df["salary_band"].map({"Low":0,"Medium":1,"High":2,"Very High":3})

le = LabelEncoder()

X = pd.get_dummies(X, columns=["sector","role_category"],dtype=int)

X["qualification_level"] = le.fit_transform(X["qualification_level"])


# 2. Splitting data
X_train, X_val, y_train, y_val = train_test_split(X, y, train_size = 0.8,test_size=0.2, random_state=42)

In [61]:
X

,qualification_level,sector_Banking,sector_Education,sector_Energy,sector_Government Admin,sector_Government General,sector_Healthcare,sector_Judiciary,sector_PSU,sector_Technical/Research,role_category_Academic,role_category_Assistant,role_category_Associate,role_category_Engineer,role_category_Entry Level,role_category_Managerial,role_category_Medical,role_category_Officer,role_category_Other,role_category_Research
0,5,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0
1,5,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0
2,5,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0
3,3,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0
4,5,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
647,5,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,0
648,2,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1
649,5,0,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0
650,5,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0


### Model Optimization 

In [62]:
model_knn = KNeighborsClassifier()

In [63]:
model_knn.fit(X_train,y_train)

,n_neighbors,5
,weights,'uniform'
,algorithm,'auto'
,leaf_size,30
,p,2
,metric,'minkowski'
,metric_params,None
,n_jobs,None


In [72]:
# predictions
train_pred = model_knn.predict_proba(X_train)
test_pred = model_knn.predict_proba(X_val)

print("Log Loss of the KNN")
print("Training_loss : ",log_loss(y_train,train_pred))
print("Validation_loss : ",log_loss(y_val,test_pred))


Log Loss of the KNN
Training_loss :  2.475337669750038
Validation_loss :  7.532500089596514


In [65]:
# predictions
train_pred = model_knn.predict(X_train)
test_pred = model_knn.predict(X_val)

# --------------------------------------------
# Validation Data

print("Validation Data loss")
print("--"*20)

print(confusion_matrix(y_val,test_pred))
print(classification_report(y_train,train_pred))
print()


# training data

print("Training Data loss")
print("--"*20)
print(confusion_matrix(y_train,train_pred))
print(classification_report(y_train,train_pred))



Validation Data loss
----------------------------------------
[[23 17  1  0]
 [25 38  4  0]
 [ 5 10  3  0]
 [ 2  1  2  0]]
              precision    recall  f1-score   support

           0       0.52      0.69      0.59       163
           1       0.58      0.67      0.62       242
           2       0.63      0.19      0.29       101
           3       0.00      0.00      0.00        15

    accuracy                           0.56       521
   macro avg       0.43      0.39      0.38       521
weighted avg       0.56      0.56      0.53       521


Training Data loss
----------------------------------------
[[112  46   5   0]
 [ 76 161   5   0]
 [ 22  60  19   0]
 [  5   9   1   0]]
              precision    recall  f1-score   support

           0       0.52      0.69      0.59       163
           1       0.58      0.67      0.62       242
           2       0.63      0.19      0.29       101
           3       0.00      0.00      0.00        15

    accuracy                    

D:\Dev\Envs\batch468\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
D:\Dev\Envs\batch468\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
D:\Dev\Envs\batch468\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
D:\Dev\Envs\batch468\Lib\site-packages\s

In [67]:
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import RandomizedSearchCV

#### Decsision Tree Classifier

#### Training Model

In [54]:
dt_model = DecisionTreeClassifier(max_depth = 14,random_state=42)

In [57]:
dt_model.fit(X_train, y_train)

,criterion,'gini'
,splitter,'best'
,max_depth,14
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,None
,random_state,42
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,class_weight,None


#### Validation 

In [73]:
# predictions
train_pred = dt_model.predict_proba(X_train)
test_pred = dt_model.predict_proba(X_val)

print("Log Loss of the decision tree")
print("Training_loss : ",log_loss(y_train,train_pred))
print("Validation_loss : ",log_loss(y_val,test_pred))



Log Loss of the decision tree
Training_loss :  0.716199991449982
Validation_loss :  7.270852470877044


In [58]:
# predictions
train_pred = dt_model.predict(X_train)
test_pred = dt_model.predict(X_val)

# --------------------------------------------
# Validation Data

print("Validation Data loss")
print("--"*20)
print(confusion_matrix(y_val,test_pred))
print(classification_report(y_train,train_pred))
print()


# training data

print("Training Data loss")
print("--"*20)
print(confusion_matrix(y_train,train_pred))
print(classification_report(y_train,train_pred))



Validation Data loss
----------------------------------------
[[19 16  6  0]
 [21 39  7  0]
 [ 4 10  4  0]
 [ 1  2  2  0]]
              precision    recall  f1-score   support

           0       0.65      0.66      0.66       163
           1       0.66      0.78      0.72       242
           2       0.59      0.43      0.49       101
           3       0.00      0.00      0.00        15

    accuracy                           0.65       521
   macro avg       0.48      0.47      0.47       521
weighted avg       0.63      0.65      0.63       521


Training Data loss
----------------------------------------
[[108  45  10   0]
 [ 39 188  15   0]
 [ 14  44  43   0]
 [  4   6   5   0]]
              precision    recall  f1-score   support

           0       0.65      0.66      0.66       163
           1       0.66      0.78      0.72       242
           2       0.59      0.43      0.49       101
           3       0.00      0.00      0.00        15

    accuracy                    

D:\Dev\Envs\batch468\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
D:\Dev\Envs\batch468\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
D:\Dev\Envs\batch468\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
D:\Dev\Envs\batch468\Lib\site-packages\s

#### Random Forest Regressor

#### Training Model

In [68]:
model_rf = RandomForestClassifier(n_estimators=150, 
    max_depth=6, 
    min_samples_leaf=5, 
    class_weight='balanced', # <--- THIS IS THE MAGIC FIX FOR CLASS 3
    random_state=42
)

In [69]:
model_rf.fit(X_train, y_train)

,n_estimators,150
,criterion,'gini'
,max_depth,6
,min_samples_split,2
,min_samples_leaf,5
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


0.884335605993645

#### Validation 

In [75]:
# predictions
train_pred = model_rf.predict_proba(X_train)
test_pred = model_rf.predict_proba(X_val)

print("Log Loss of the decision tree")
print("Training_loss : ",log_loss(y_train,train_pred))
print("Validation_loss : ",log_loss(y_val,test_pred))



Log Loss of the decision tree
Training_loss :  1.1620382943097987
Validation_loss :  1.276505247384932


In [86]:
# predictions
train_pred = model_rf.predict(X_train)
test_pred = model_rf.predict(X_val)

# --------------------------------------------
# Validation Data

print("Validation Data loss")
print("--"*20)
print(confusion_matrix(y_val,test_pred))
print(classification_report(y_val,test_pred))
print()


# training data

print("Training Data loss")
print("--"*20)
print(confusion_matrix(y_train,train_pred))
print(classification_report(y_train,train_pred))



Validation Data loss
----------------------------------------
[[24  9  4  4]
 [22 17 18 10]
 [ 2  5  5  6]
 [ 1  0  4  0]]
              precision    recall  f1-score   support

           0       0.49      0.59      0.53        41
           1       0.55      0.25      0.35        67
           2       0.16      0.28      0.20        18
           3       0.00      0.00      0.00         5

    accuracy                           0.35       131
   macro avg       0.30      0.28      0.27       131
weighted avg       0.46      0.35      0.37       131


Training Data loss
----------------------------------------
[[ 97  32  19  15]
 [ 52 110  51  29]
 [ 10  17  46  28]
 [  4   1   2   8]]
              precision    recall  f1-score   support

           0       0.60      0.60      0.60       163
           1       0.69      0.45      0.55       242
           2       0.39      0.46      0.42       101
           3       0.10      0.53      0.17        15

    accuracy                    

In [77]:
from sklearn.ensemble import AdaBoostClassifier, GradientBoostingClassifier

In [78]:
model_gb = GradientBoostingClassifier(n_estimators=100, max_depth=3, random_state=42)
model_gb.fit(X_train,y_train)

,loss,'log_loss'
,learning_rate,0.1
,n_estimators,100
,subsample,1.0
,criterion,'friedman_mse'
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_depth,3
,min_impurity_decrease,0.0
,init,None


#### Validation 

In [83]:
# predictions
train_pred = model_gb.predict_proba(X_train)
test_pred = model_gb.predict_proba(X_val)

print("Log Loss of the gradient Boosting")
print("Training_loss : ",log_loss(y_train,train_pred))
print("Validation_loss : ",log_loss(y_val,test_pred))



Log Loss of the gradient Boosting
Training_loss :  0.8224622629376762
Validation_loss :  1.1462790683060269


In [85]:
# predictions
train_pred = model_gb.predict(X_train)
test_pred = model_gb.predict(X_val)

# --------------------------------------------
# Validation Data

print("Validation Data loss")
print("--"*20)
print(confusion_matrix(y_val,test_pred))
print(classification_report(y_val,test_pred))
print()


# training data

print("Training Data loss")
print("--"*20)
print(confusion_matrix(y_train,train_pred))
print(classification_report(y_train,train_pred))



Validation Data loss
----------------------------------------
[[17 22  2  0]
 [12 49  6  0]
 [ 0 12  6  0]
 [ 1  3  1  0]]
              precision    recall  f1-score   support

           0       0.57      0.41      0.48        41
           1       0.57      0.73      0.64        67
           2       0.40      0.33      0.36        18
           3       0.00      0.00      0.00         5

    accuracy                           0.55       131
   macro avg       0.38      0.37      0.37       131
weighted avg       0.52      0.55      0.53       131


Training Data loss
----------------------------------------
[[ 91  67   5   0]
 [ 27 198  16   1]
 [  6  58  36   1]
 [  2   7   4   2]]
              precision    recall  f1-score   support

           0       0.72      0.56      0.63       163
           1       0.60      0.82      0.69       242
           2       0.59      0.36      0.44       101
           3       0.50      0.13      0.21        15

    accuracy                    

D:\Dev\Envs\batch468\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
D:\Dev\Envs\batch468\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
D:\Dev\Envs\batch468\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [87]:
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import classification_report, confusion_matrix

In [91]:
# 1. Compute your sample weights directly on your hand-encoded y_train
sample_weights = compute_sample_weight(class_weight='balanced', y=y_train)

# 2. Define the grid of parameters to test
param_grid = {
    'n_estimators': [100, 150, 200],      # Give it more trees to learn
    'learning_rate': [0.05, 0.1, 0.2],     # Faster learning speeds
    'max_depth': [3, 4, 5],                # Allow slightly deeper trees if needed
    'min_samples_leaf': [3, 5, 10]
}
# 3. Initialize the base Gradient Boosting model variable
gb_base_model = GradientBoostingClassifier(random_state=42)

# 4. Set up the Grid Search 
grid_search = GridSearchCV(
    estimator=gb_base_model, 
    param_grid=param_grid, 
    cv=5, 
    scoring='f1_macro', 
    n_jobs=-1,
    verbose=True
)

# 5. Run the search directly on your pre-encoded variables
grid_search.fit(X_train, y_train, sample_weight=sample_weights)

# 6. Extract the best parameters and the best trained model variable
print("Best Parameters Found:")
print(grid_search.best_params_)

best_gb_model = grid_search.best_estimator_

# 7. Evaluate the final optimized model variable
y_train_pred = best_gb_model.predict(X_train)
y_val_pred = best_gb_model.predict(X_val)

print("\n=== Optimized Validation Data Loss ===")
print(confusion_matrix(y_val, y_val_pred))
print(classification_report(y_val, y_val_pred))

print("\n=== Optimized Training Data Loss ===")
print(confusion_matrix(y_train, y_train_pred))
print(classification_report(y_train, y_train_pred))

Fitting 5 folds for each of 81 candidates, totalling 405 fits
Best Parameters Found:
{'learning_rate': 0.2, 'max_depth': 4, 'min_samples_leaf': 10, 'n_estimators': 100}

=== Optimized Validation Data Loss ===
[[19 10  4  8]
 [14 23 17 13]
 [ 2  6  3  7]
 [ 1  0  4  0]]
              precision    recall  f1-score   support

           0       0.53      0.46      0.49        41
           1       0.59      0.34      0.43        67
           2       0.11      0.17      0.13        18
           3       0.00      0.00      0.00         5

    accuracy                           0.34       131
   macro avg       0.31      0.24      0.26       131
weighted avg       0.48      0.34      0.39       131


=== Optimized Training Data Loss ===
[[ 89  28  19  27]
 [ 34 123  46  39]
 [  4  13  50  34]
 [  0   1   0  14]]
              precision    recall  f1-score   support

           0       0.70      0.55      0.61       163
           1       0.75      0.51      0.60       242
           2     

In [96]:
# Revert to your best-performing baseline setup
best_fit_model = GradientBoostingClassifier(
    learning_rate=0.2, 
    n_estimators=100, 
    max_depth=4, 
    min_samples_leaf=10,
    random_state=42
)
best_fit_model.fit(X_train,y_train)

,loss,'log_loss'
,learning_rate,0.2
,n_estimators,100
,subsample,1.0
,criterion,'friedman_mse'
,min_samples_split,2
,min_samples_leaf,10
,min_weight_fraction_leaf,0.0
,max_depth,4
,min_impurity_decrease,0.0
,init,None


#### Validation 

In [97]:
# predictions
train_pred = best_fit_model.predict_proba(X_train)
test_pred = best_fit_model.predict_proba(X_val)

print("Log Loss of the gradient Boosting")
print("Training_loss : ",log_loss(y_train,train_pred))
print("Validation_loss : ",log_loss(y_val,test_pred))



Log Loss of the gradient Boosting
Training_loss :  0.8217604879045459
Validation_loss :  1.2673666653644278


In [98]:
# predictions
train_pred = best_fit_model.predict(X_train)
test_pred = best_fit_model.predict(X_val)

# --------------------------------------------
# Validation Data

print("Validation Data loss")
print("--"*20)
print(confusion_matrix(y_val,test_pred))
print(classification_report(y_val,test_pred))
print()


# training data

print("Training Data loss")
print("--"*20)
print(confusion_matrix(y_train,train_pred))
print(classification_report(y_train,train_pred))



Validation Data loss
----------------------------------------
[[18 20  3  0]
 [11 48  8  0]
 [ 1 10  7  0]
 [ 1  3  1  0]]
              precision    recall  f1-score   support

           0       0.58      0.44      0.50        41
           1       0.59      0.72      0.65        67
           2       0.37      0.39      0.38        18
           3       0.00      0.00      0.00         5

    accuracy                           0.56       131
   macro avg       0.39      0.39      0.38       131
weighted avg       0.54      0.56      0.54       131


Training Data loss
----------------------------------------
[[ 87  66  10   0]
 [ 23 195  23   1]
 [  6  49  45   1]
 [  2   5   6   2]]
              precision    recall  f1-score   support

           0       0.74      0.53      0.62       163
           1       0.62      0.81      0.70       242
           2       0.54      0.45      0.49       101
           3       0.50      0.13      0.21        15

    accuracy                    

D:\Dev\Envs\batch468\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
D:\Dev\Envs\batch468\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
D:\Dev\Envs\batch468\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
